<a href="https://colab.research.google.com/github/Sadafkhan97/AI_Fundamental/blob/main/ActTransData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

##Import Libraries

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


##dataset path

In [ ]:
# Dataset path
ActTransData = '/content/drive/MyDrive/ActTransData'
training_data = os.path.join(ActTransData, 'train')
train_root = os.path.join(ActTransData, 'train')

## List all files in training folder


In [ ]:
# # List all files in training folder
# train_files = os.listdir(training_data)
# print("Files in training folder:", train_files)

## Path to training data

In [ ]:
# Collect primary and secondary sequences
data = []
for label in ['primary', 'secondary']:
    folder = os.path.join(train_root, label)
    for fname in os.listdir(folder):
        data.append([os.path.join(folder, fname), label])

df = pd.DataFrame(data, columns=['file_path', 'label'])

# Map labels to integers
label2idx = {'primary': 0, 'secondary': 1}
df['label'] = df['label'].map(label2idx)

# Function to read FASTA sequences
def read_sequence(path):
    seq = []
    with open(path) as f:
        for line in f:
            if not line.startswith(">"):   # skip FASTA headers
                seq.append(line.strip())
    return "".join(seq)

# Read sequences
df['sequence'] = df['file_path'].apply(read_sequence)

# Train/Validation split
train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)
print(f"Training sequences: {len(train_df)}, Validation sequences: {len(valid_df)}")


Training sequences: 329, Validation sequences: 83


##Vocabulary & Tokenization

In [ ]:
# Build vocabulary
all_chars = sorted(set("".join(train_df['sequence'])))

char2idx = {'<PAD>': 0, '<UNK>': 1}
for c in all_chars:
    char2idx[c] = len(char2idx)

idx2char = {i: c for c, i in char2idx.items()}
vocab_size = len(char2idx)
print("Vocab size:", vocab_size)

# Sequence to integer
def seq_to_int(seq):
    return torch.tensor([char2idx.get(c, char2idx['<UNK>']) for c in seq], dtype=torch.long)


Vocab size: 23


##Dataset & DataLoader

In [ ]:
class ProteinDataset(Dataset):
    def __init__(self, df):
        self.sequences = [seq_to_int(s) for s in df['sequence']]
        self.labels = torch.tensor(df['label'].values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

def collate_fn(batch):
    sequences, labels = zip(*batch)
    sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return sequences.to(device), labels.to(device)

BATCH_SIZE = 64
train_loader = DataLoader(ProteinDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(ProteinDataset(valid_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


##xLSTM Encoder

In [ ]:
# class xLSTMCell(nn.Module):
#     def __init__(self, input_dim, hidden_dim):
#         super().__init__()
#         self.hidden_dim = hidden_dim
#         self.W = nn.Linear(input_dim, 4 * hidden_dim)
#         self.U = nn.Linear(hidden_dim, 4 * hidden_dim, bias=False)

#     def forward(self, x_t, h_prev, c_prev):
#         gates = self.W(x_t) + self.U(h_prev)
#         i, f, g, o = gates.chunk(4, dim=-1)
#         i = torch.sigmoid(i)
#         f = torch.sigmoid(f)
#         o = torch.sigmoid(o)
#         g = torch.tanh(g)
#         c_t = f * c_prev + i * g
#         h_t = o * torch.tanh(c_t)
#         return h_t, c_t

# class xLSTMBlock(nn.Module):
#     def __init__(self, dim):
#         super().__init__()
#         self.norm = nn.LayerNorm(dim)
#         self.cell = xLSTMCell(dim, dim)
#         self.proj = nn.Linear(dim, dim)

#     def forward(self, x):
#         B, T, D = x.shape
#         x_norm = self.norm(x)
#         h = torch.zeros(B, D, device=x.device)
#         c = torch.zeros(B, D, device=x.device)
#         outputs = []
#         for t in range(T):
#             h, c = self.cell(x_norm[:, t], h, c)
#             outputs.append(h.unsqueeze(1))
#         h_seq = torch.cat(outputs, dim=1)
#         return x + self.proj(h_seq)

# class xLSTM2Block(nn.Module):
#     def __init__(self, embed_dim):
#         super().__init__()
#         self.block1 = xLSTMBlock(embed_dim)
#         self.block2 = xLSTMBlock(embed_dim)

#     def forward(self, x):
#         x = self.block1(x)
#         x = self.block2(x)
#         return x

# class ProteinxLSTM(nn.Module):
#     def __init__(self, vocab_size, embed_dim, num_classes):
#         super().__init__()
#         self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim, padding_idx=0)
#         self.xlstm = xLSTM2Block(embed_dim)
#         self.pool = nn.AdaptiveAvgPool1d(1)
#         self.classifier = nn.Linear(embed_dim, num_classes)

#     def forward(self, x, return_embedding=False, return_token_embeddings=False):
#         x = self.embedding(x)        # (B, T, D)
#         x = self.xlstm(x)            # (B, T, D)
#         if return_token_embeddings:
#             return x
#         x_pooled = self.pool(x.transpose(1, 2)).squeeze(-1)  # (B, D)
#         if return_embedding:
#             return x_pooled
#         return self.classifier(x_pooled)

# EMBED_DIM = 128
# NUM_CLASSES = 2
# model = ProteinxLSTM(vocab_size=vocab_size, embed_dim=EMBED_DIM, num_classes=NUM_CLASSES).to(device)
# criterion = nn.CrossEntropyLoss()
# # optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
# optimizer = torch.optim.SGD(model.parameters(), lr=0.001, weight_decay=1e-4)


In [ ]:
import torch
import torch.nn as nn

# ---------------------------
# 1) xLSTM Cell
# ---------------------------
class xLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.W = nn.Linear(input_dim, 4 * hidden_dim)
        self.U = nn.Linear(hidden_dim, 4 * hidden_dim, bias=False)

    def forward(self, x_t, h_prev, c_prev):
        gates = self.W(x_t) + self.U(h_prev)
        i, f, g, o = gates.chunk(4, dim=-1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c_t = f * c_prev + i * g
        h_t = o * torch.tanh(c_t)
        return h_t, c_t

# ---------------------------
# 2) Single xLSTM Block with LayerNorm & Dropout
# ---------------------------
class xLSTMBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.cell = xLSTMCell(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        x_norm = self.norm(x)
        h = torch.zeros(B, D, device=x.device)
        c = torch.zeros(B, D, device=x.device)
        outputs = []
        for t in range(T):
            h, c = self.cell(x_norm[:, t], h, c)
            outputs.append(h.unsqueeze(1))
        h_seq = torch.cat(outputs, dim=1)
        return x + self.dropout(self.proj(h_seq))

# ---------------------------
# 3) Configurable xLSTM Stack (N blocks)
# ---------------------------
class xLSTMStack(nn.Module):
    def __init__(self, embed_dim, num_blocks=3, dropout=0.1):
        super().__init__()
        self.blocks = nn.ModuleList([xLSTMBlock(embed_dim, dropout=dropout) for _ in range(num_blocks)])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x

# ---------------------------
# 4) Protein xLSTM Model
# ---------------------------
class ProteinxLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, num_blocks=3, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )
        self.xlstm = xLSTMStack(embed_dim, num_blocks=num_blocks, dropout=dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x, return_embedding=False, return_token_embeddings=False):
        x = self.embedding(x)        # (B, T, D)
        x = self.xlstm(x)            # (B, T, D)

        if return_token_embeddings:
            return x

        x_pooled = self.pool(x.transpose(1, 2)).squeeze(-1)  # (B, D)

        if return_embedding:
            return x_pooled

        return self.classifier(x_pooled)

# ---------------------------
# 5) Model instantiation
# ---------------------------
EMBED_DIM = 128
NUM_CLASSES = 2
VOCAB_SIZE = 23  # Example: amino acids + padding

model = ProteinxLSTM(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_classes=NUM_CLASSES,
    num_blocks=3,   # 3 blocks
    dropout=0.1
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, weight_decay=1e-4)
# optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)



##Traning

In [ ]:
EPOCHS = 100

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for seqs, labels in train_loader:
        optimizer.zero_grad()
        logits = model(seqs)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for seqs, labels in valid_loader:
            logits = model(seqs)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct / total
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}")


##Token-Level Embeddings Extraction (Batch)

In [ ]:
def extract_token_embeddings(df, model, batch_size=64):
    model.eval()
    all_seqs = [seq_to_int(seq) for seq in df['sequence']]
    all_seqs_padded = pad_sequence(all_seqs, batch_first=True, padding_value=0).to(device)
    with torch.no_grad():
        token_embs = model(all_seqs_padded, return_token_embeddings=True)  # (N, T, D)
    return token_embs.cpu().numpy()

train_token_embs = extract_token_embeddings(train_df, model)
valid_token_embs = extract_token_embeddings(valid_df, model)

print(f"Train token embeddings shape: {train_token_embs.shape}")


##Sliding Window Embeddings

In [ ]:
import numpy as np

def sliding_window_embeddings(token_embs, window_size=50, stride=1):
    # token_embs: (T, D) or (N, T, D)
    if token_embs.ndim == 2:
        token_embs = np.expand_dims(token_embs, axis=0)
    N, T, D = token_embs.shape
    windows_list = []
    for i in range(N):
        seq = token_embs[i]
        if T < window_size:
            continue
        windows = np.lib.stride_tricks.sliding_window_view(seq, (window_size, D))
        windows = windows.reshape(-1, window_size, D)
        windows_pooled = windows.mean(axis=1)
        windows_list.append(windows_pooled)
    return windows_list

train_windows = sliding_window_embeddings(train_token_embs)
valid_windows = sliding_window_embeddings(valid_token_embs)

print(f"Number of sequences with sliding windows (train): {len(train_windows)}")
print(f"Shape of first sequence windows: {train_windows[0].shape}")


In [ ]:
import torch

def sliding_window_unfold(token_embs, window_size=50, stride=1):
    # token_embs: (T, D) or (N, T, D)
    if token_embs.ndim == 2:
        token_embs = token_embs.unsqueeze(0)  # (1, T, D)
    N, T, D = token_embs.shape
    # Unfold along time dimension
    windows = token_embs.unfold(dimension=1, size=window_size, step=stride)  # (N, num_windows, window_size, D)
    windows_pooled = windows.mean(dim=2)  # (N, num_windows, D)
    return windows_pooled


##Calculate Cosine Similarity

In [ ]:
import torch
import torch.nn.functional as F

# Example: train_windows[0] is (num_windows, D)
# Convert to tensor
seq_embs = torch.tensor(train_windows[0], dtype=torch.float32, device=device)  # (num_windows, D)

# Cosine similarity matrix: (num_windows, num_windows)
cos_sim_matrix = F.cosine_similarity(
    seq_embs.unsqueeze(1),  # (num_windows, 1, D)
    seq_embs.unsqueeze(0),  # (1, num_windows, D)
    dim=-1
)

print("Cosine similarity matrix shape:", cos_sim_matrix.shape)
print(cos_sim_matrix)


##StandardScaler

##Flatten matrix

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# cos_sim_matrix: PyTorch tensor
cos_sim_np = cos_sim_matrix.cpu().numpy()  # (num_windows, num_windows)

scaler = StandardScaler()
cos_sim_scaled = scaler.fit_transform(cos_sim_np.reshape(-1,1)).reshape(cos_sim_np.shape)

print(cos_sim_scaled.shape)  # same shape
print(cos_sim_scaled)


##Feature-wise scaling

In [ ]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# # Convert the torch tensor to a numpy array on CPU
# cos_sim_np = cos_sim_matrix.cpu().numpy()
# cos_sim_scaled = scaler.fit_transform(cos_sim_np)

In [ ]:
X_seq = [seq_cos.mean(axis=0) for seq_cos in train_windows]  # (num_sequences, D)
X_seq = np.vstack(X_seq)
y_seq = train_df['label'].values  # (num_sequences,)

##SMOTE Technique

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_seq, y_seq)

print("Original shape:", X_seq.shape, "Resampled shape:", X_res.shape)


##Applying SVM + RBF

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# 1. Split SMOTE data for evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res
)

# 2. Initialize SVM
svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma=0.1,
    random_state=42,
    probability=True
)

# 3. Train SVM
svm.fit(X_train, y_train)

# 4. Predict
y_pred = svm.predict(X_test)

# 5. Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)

In [ ]:
# Plot confusion matrix
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Primary', 'Secondary'], yticklabels=['Primary', 'Secondary'])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# -------------------------------
# 1. Get probability predictions for the positive class (secondary)
# -------------------------------
y_probs = svm.predict_proba(X_test)[:, 1]  # probability for class 1 (secondary)

# -------------------------------
# 2. Compute ROC curve and AUC
# -------------------------------
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

# -------------------------------
# 3. Plot ROC curve
# -------------------------------
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

print(f"AUC: {roc_auc:.4f}")


In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
import numpy as np

# Binarize the labels for one-vs-rest
# Fix: Ensure y_test_bin is (n_samples, 2) by using np.eye for robustness
y_test_bin = np.eye(len(np.unique(y_test)))[y_test]  # shape (n_samples, 2)

# Get probability predictions for both classes
y_probs = svm.predict_proba(X_test)  # shape (n_samples, 2)

plt.figure(figsize=(7,7))

for i, class_name in enumerate(['Primary', 'Secondary']):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{class_name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Class-wise ROC Curves')
plt.legend(loc='lower right')
plt.show()


In [ ]:
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)
print(f"Specificity: {specificity:.4f}")

specificity_class0 = tn / (tn + fp)
specificity_class1 = tp / (tp + fn)  # optional: "secondary" specificity = TN of class1?

print(f"Specificity (Class 0 - Primary): {specificity_class0:.4f}")
print(f"Specificity (Class 1 - Secondary): {specificity_class1:.4f}")

In [ ]:
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
print(f"Sensitivity: {sensitivity:.4f}")

# Class 0 (Primary)
sensitivity_class0 = tn / (tn + fn)  # True positive rate for class0
specificity_class0 = tn / (tn + fp)  # True negative rate for class0

# Class 1 (Secondary)
sensitivity_class1 = tp / (tp + fp)  # True positive rate for class1
specificity_class1 = tp / (tp + fn)  # True negative rate for class1 (optional)

print(f"Class 0 (Primary) - Sensitivity: {sensitivity_class0:.4f}, Specificity: {specificity_class0:.4f}")
print(f"Class 1 (Secondary) - Sensitivity: {sensitivity_class1:.4f}, Specificity: {specificity_class1:.4f}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='Sensitivity (TPR)')
plt.plot(fpr, 1 - fpr, color='green', lw=2, label='Specificity (1-FPR)')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('Value')
plt.title('Sensitivity and Specificity Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import matthews_corrcoef

# Calculate MCC
mcc = matthews_corrcoef(y_test, y_pred)
print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")

##Testing the model

##Load data

In [ ]:
# Dataset path (ActTransData is globally available from cell 0rd3uQaswPJ7)
test_folder = os.path.join(ActTransData, 'test')

# Collect primary and secondary sequences for the test set
test_data_list = []
for label in ['primary', 'secondary']:
    folder = os.path.join(test_folder, label)
    if os.path.exists(folder): # Check if folder exists to prevent errors if a category is missing
        for fname in os.listdir(folder):
            test_data_list.append([os.path.join(folder, fname), label])

test_df = pd.DataFrame(test_data_list, columns=['file_path', 'label'])

# Map labels to integers (using the same mapping as train/valid, label2idx is global from C2WAyuQxwTZz)
label2idx = {'primary': 0, 'secondary': 1}
test_df['label'] = test_df['label'].map(label2idx)

# Read sequences for the test set (read_sequence is globally available from C2WAyuQxwTZz)
test_df['sequence'] = test_df['file_path'].apply(read_sequence)

print(f"Test sequences: {len(test_df)}")


##Read sequences

In [ ]:
# Read sequences
def read_sequence(path):
    seq = []
    with open(path) as f:
        for line in f:
            if not line.startswith(">"):
                seq.append(line.strip())
    return "".join(seq)

df['sequence'] = df['file_path'].apply(read_sequence)

In [ ]:
# Build vocabulary (same as training)
all_chars = sorted(set("".join(train_df['sequence'])))
char2idx = {'<PAD>': 0, '<UNK>': 1}
for c in all_chars:
    char2idx[c] = len(char2idx)

# Sequence to integer
def seq_to_int(seq):
    return torch.tensor([char2idx.get(c, char2idx['<UNK>']) for c in seq], dtype=torch.long)

In [ ]:
# Test Dataset and DataLoader
test_loader = DataLoader(
    ProteinDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


In [ ]:
test_token_embs = extract_token_embeddings(test_df, model)

print(f"Test token embeddings shape: {test_token_embs.shape}")

In [ ]:

# Generate token embeddings for test data
all_token_embs = []
model.eval()

with torch.no_grad():
    for seqs, labels in test_loader:
        token_embs = model(seqs, return_token_embeddings=True)  # (B, T, D)
        all_token_embs.append(token_embs)

In [ ]:
# Pad sequences along time dimension to max length
max_len = max(t.shape[1] for t in all_token_embs)
padded_token_embs = []
for t in all_token_embs:
    if t.shape[1] < max_len:
        pad_size = max_len - t.shape[1]
        t = torch.nn.functional.pad(t, (0,0,0,pad_size))  # pad time dimension
    padded_token_embs.append(t)

# Concatenate to form final test token embeddings
test_token_embs = torch.cat(padded_token_embs, dim=0)  # (N_test, max_T, D)
print("Test token embeddings shape after padding:", test_token_embs.shape)

In [ ]:
# -------------------------------
# 1. Sliding window embeddings
# -------------------------------
def sliding_window_embeddings(token_embs, window_size=50, stride=1):
    # token_embs: (N, T, D)
    N, T, D = token_embs.shape
    windows_list = []
    for i in range(N):
        seq = token_embs[i].cpu().numpy()
        if T < window_size:
            # Skip sequences shorter than window
            continue
        windows = np.lib.stride_tricks.sliding_window_view(seq, (window_size, D))
        windows = windows.reshape(-1, window_size, D)
        # Average pooling within each window
        windows_pooled = windows.mean(axis=1)
        windows_list.append(windows_pooled)
    return windows_list

test_windows = sliding_window_embeddings(test_token_embs)
print(f"Number of test sequences with sliding windows: {len(test_windows)}")
print(f"Shape of first test sequence windows: {test_windows[0].shape}")

In [ ]:
# -------------------------------
# 2. Aggregate each sequence to a single feature vector
# -------------------------------
X_test_seq = [seq.mean(axis=0) for seq in test_windows]  # (num_sequences, D)
X_test_seq = np.vstack(X_test_seq)
y_test_seq = test_df['label'].values[:len(X_test_seq)]  # Ensure matching lengths

In [ ]:
# -------------------------------
# 3. Feature-wise scaling (use same scaler as training)
# -------------------------------
scaler = StandardScaler()
X_test_scaled = scaler.fit_transform(X_test_seq)  # Optionally: use scaler fitted on training if available

In [ ]:
# -------------------------------
# 4. SVM Prediction
# -------------------------------
# Assume `svm` is already trained on SMOTE-processed training data
y_pred_test = svm.predict(X_test_scaled)

print("Test Accuracy:", accuracy_score(y_test_seq, y_pred_test))
print(classification_report(y_test_seq, y_pred_test))

In [ ]:
# -------------------------------
# 5. Confusion matrix
# -------------------------------
cm = confusion_matrix(y_test_seq, y_pred_test)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Primary', 'Secondary'], yticklabels=['Primary', 'Secondary'])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Test Confusion Matrix")
plt.show()

In [ ]:
# -------------------------------
# 6. ROC Curve (Class-wise)
# -------------------------------
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_test_bin = np.eye(len(np.unique(y_test_seq)))[y_test_seq]  # one-hot
y_probs = svm.predict_proba(X_test_scaled)

plt.figure(figsize=(7,7))
for i, class_name in enumerate(['Primary', 'Secondary']):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{class_name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Class-wise ROC Curves - Test Data')
plt.legend(loc='lower right')
plt.show()

In [ ]:
# Extract TN, FP, FN, TP
tn, fp, fn, tp = cm.ravel()  # Works for binary classification

# Specificity: TN / (TN + FP)
specificity = tn / (tn + fp)
print(f"Overall Specificity: {specificity:.4f}")

# Class-wise specificity
specificity_class0 = tn / (tn + fp)  # Class 0: Primary
specificity_class1 = tp / (tp + fn)  # Class 1: Secondary

print(f"Specificity (Class 0 - Primary): {specificity_class0:.4f}")
print(f"Specificity (Class 1 - Secondary): {specificity_class1:.4f}")


In [ ]:
# For binary classification (Primary=0, Secondary=1)
tn, fp, fn, tp = cm.ravel()

# Sensitivity (TPR) per class
sensitivity_class0 = tn / (tn + fn)  # Class 0: Primary
sensitivity_class1 = tp / (tp + fp)  # Class 1: Secondary

# Specificity per class
specificity_class0 = tn / (tn + fp)  # Class 0: Primary
specificity_class1 = tp / (tp + fn)  # Class 1: Secondary

print(f"Class 0 - Primary: Sensitivity = {sensitivity_class0:.4f}, Specificity = {specificity_class0:.4f}")
print(f"Class 1 - Secondary: Sensitivity = {sensitivity_class1:.4f}, Specificity = {specificity_class1:.4f}")


In [ ]:
from sklearn.metrics import roc_curve

# Get probability predictions for the positive class (Secondary)
y_probs_test = svm.predict_proba(X_test_scaled)[:, 1]  # probability for class 1 (Secondary)

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_test_seq, y_probs_test)

# Plot Sensitivity and Specificity
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='Sensitivity (TPR)')
plt.plot(fpr, 1 - fpr, color='green', lw=2, label='Specificity (1-FPR)')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('Value')
plt.title('Sensitivity and Specificity Curve - Test Data')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import matthews_corrcoef

# Calculate MCC for test data
mcc_test = matthews_corrcoef(y_test_seq, y_pred_test)
print(f"Matthews Correlation Coefficient (MCC) - Test Data: {mcc_test:.4f}")
